# 04. A/B Test Analysis

Разбираем тест `free_delivery_banner_v1`: сравниваем `control` и `treatment` по основной метрике `7-day conversion to delivered order`.

Дальше считаем `uplift`, `z-test`, `p-value` и грубую оценку бизнес-эффекта в `gross margin RUB`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import norm

orders = pd.read_csv(Path('../data/fact_orders.csv'), parse_dates=['order_ts'])
ab = pd.read_csv(Path('../data/fact_ab_test_assignments.csv'), parse_dates=['assigned_at'])
orders = orders[orders['order_status'] == 'delivered'].copy()

In [2]:
joined = ab.merge(orders[['user_id', 'order_ts']], on='user_id', how='left')
joined['within_7d'] = (joined['order_ts'] >= joined['assigned_at']) & (joined['order_ts'] < joined['assigned_at'] + pd.Timedelta(days=7))
user_conv = joined.groupby(['user_id', 'variant'], as_index=False)['within_7d'].max()
summary = user_conv.groupby('variant').agg(
    users=('user_id', 'count'),
    converted=('within_7d', 'sum')
)
summary['conversion'] = summary['converted'] / summary['users']

summary_display = summary.copy()
summary_display['conversion'] = (summary_display['conversion'] * 100).round(2).astype(str) + '%'
summary_display.style.hide(axis='index')


users,converted,conversion
2857,1697,59.4%
2792,1626,58.24%


In [3]:
p_c = float(summary.loc['control', 'conversion'])
p_t = float(summary.loc['treatment', 'conversion'])
n_c = int(summary.loc['control', 'users'])
n_t = int(summary.loc['treatment', 'users'])

p_pool = float(summary['converted'].sum() / summary['users'].sum())
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_c + 1 / n_t))
z = float((p_t - p_c) / se) if se else 0.0
p_value = float(2 * (1 - norm.cdf(abs(z))))
uplift_pp = (p_t - p_c) * 100
is_significant = p_value < 0.05

test_result = pd.DataFrame({
    'metric': ['Control conversion', 'Treatment conversion', 'Uplift', 'z-stat', 'p-value', 'Result'],
    'value': [
        f'{p_c:.2%}',
        f'{p_t:.2%}',
        f'{uplift_pp:+.2f} pp',
        f'{z:.2f}',
        f'{p_value:.3f}',
        'Statistically significant' if is_significant else 'Not statistically significant',
    ]
})
test_result.style.hide(axis='index')


metric,value
Control conversion,59.40%
Treatment conversion,58.24%
Uplift,-1.16 pp
z-stat,-0.89
p-value,0.376
Result,Not statistically significant


In [4]:
avg_margin = float(orders['gross_margin_rub'].mean())
incremental_users = uplift_pp / 100 * n_t
incremental_margin_30d = incremental_users * 1.8 * avg_margin

business_impact = pd.DataFrame({
    'metric': ['Incremental users', 'Incremental gross margin 30D', 'Annualized gross margin'],
    'value': [
        f'{incremental_users:.0f}',
        f'{incremental_margin_30d:,.0f} RUB',
        f'{incremental_margin_30d * 12:,.0f} RUB',
    ]
})
business_impact.style.hide(axis='index')


metric,value
Incremental users,-32
Incremental gross margin 30D,"-14,400 RUB"
Annualized gross margin,"-172,799 RUB"
